# Analyse de l'entraînement mTAN (LSST, 4 bandes)

Ce notebook lit le **log SLURM** (`mTan_train_<jobid>.out`) produit par `tan_unsupervised_lsst.py` et trace :

1. MSE d'entraînement / validation / test (avec l'axe équivalent en magnitudes)
2. Perte (ELBO IWAE), reconstruction et KL, avec le calendrier d'annealing du KL
3. Diagnostic de surapprentissage (courbes lissées, rapport val/train, tableau par fenêtres, pentes)
4. Poids relatif du terme KL
5. Choix du meilleur checkpoint (bruit de la validation)
6. *(optionnel)* Analyse par courbe à partir des sorties de `evaluate_unsupervised_lsst.py`

Il n'utilise que `numpy`, `pandas` et `matplotlib` (pas de torch, pas de GPU).

**Rappels de lecture.** Le MSE est en unités normalisées : `(mag - mag_min) / 2.5` par courbe, donc `RMSE_mag = 2.5 * sqrt(MSE)`.
Le MSE d'entraînement est mesuré avec l'augmentation de troncature et la moyenne de `k_iwae` tirages, alors que celui de validation/test
utilise des courbes complètes et un seul tirage : **les deux niveaux ne sont pas directement comparables, seules les tendances le sont.**

## 0. Paramètres

In [ ]:
import bisect, glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ------------------------- À ADAPTER -------------------------
LOG_FILE = None   # ex: "mTan_train_575402.out" ; None -> le .out le plus récent du dossier
TAG = "ftransfer_lsst_2026_06_29_882968"   # pour la section 6 (fichiers evaluate_*_<TAG>.*)
SMOOTH = 10       # fenêtre de lissage, en epochs
WINDOW = 50       # taille des fenêtres du tableau récapitulatif, en epochs
FIG_DIR = "figures"
# --------------------------------------------------------------

os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "figure.constrained_layout.use": True, "axes.grid": True, "grid.alpha": 0.3, "legend.fontsize": 9})

if LOG_FILE is None:
    cands = sorted(glob.glob("mTan_train_*.out"), key=os.path.getmtime)
    if not cands:
        raise FileNotFoundError("Aucun mTan_train_*.out dans le dossier courant : renseigne LOG_FILE.")
    LOG_FILE = cands[-1]
print("Log analysé :", LOG_FILE)

def save(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    print("figure enregistrée :", path)

## 1. Lecture du log

In [ ]:
text = open(LOG_FILE, errors="replace").read()

num = r"([-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?|nan|inf)"
it_re = re.compile(rf"Iter:\s*(\d+),\s*avg elbo:\s*{num},\s*avg reconst:\s*{num},\s*avg kl:\s*{num},\s*mse:\s*{num},\s*val_metric:\s*{num}")
te_re = re.compile(rf"Test Mean Squared Error\s*tensor\(\s*{num}")
sv_re = re.compile(r"Saving the model at epoch\s+(\d+)")

rows, starts = [], []
for m in it_re.finditer(text):
    rows.append([int(m.group(1))] + [float(m.group(i)) for i in range(2, 7)])
    starts.append(m.start())
df = pd.DataFrame(rows, columns=["epoch", "elbo", "reconst", "kl", "mse", "val"])
if df.empty:
    raise ValueError("Aucune ligne 'Iter:' trouvée : vérifie le fichier de log.")

# tests (tous les 5 epochs) : rattachés à l'epoch qui les précède dans le log
tests = []
for m in te_re.finditer(text):
    k = bisect.bisect_left(starts, m.start()) - 1
    if k >= 0:
        tests.append((rows[k][0], float(m.group(1))))
test = pd.DataFrame(tests, columns=["epoch", "test"])

saved = [int(x) for x in sv_re.findall(text)]
best_epoch = saved[-1] if saved else int(df.loc[df.val.idxmin(), "epoch"])

# doublons éventuels (log concaténés) : on garde la dernière occurrence
df = df.drop_duplicates("epoch", keep="last").sort_values("epoch").reset_index(drop=True)
test = test.drop_duplicates("epoch", keep="last")

# options de l'entraînement, lues dans la ligne Namespace(...)
def opt(name, cast=float, default=None):
    m = re.search(rf"\b{name}=([^,\s)]+)", text)
    return cast(m.group(1)) if m else default
use_kl = (re.search(r"\bkl=(True|False)", text) or [None, "False"])[1] == "True"
k_iwae = opt("k_iwae", int); lr = opt("lr"); latent = opt("latent_dim", int)

def kl_coef(itr, wait=10):   # même formule que tan_unsupervised.py
    if not use_kl:
        return 1.0
    return 0.0 if itr < wait else 1 - 0.99 ** (itr - wait)
df["kl_coef"] = df.epoch.map(kl_coef)

print(f"{len(df)} epochs lus ; {len(test)} évaluations test ; options : lr={lr}, k_iwae={k_iwae}, latent_dim={latent}, --kl={use_kl}")
print(f"Meilleur checkpoint (dernier 'Saving the model') : epoch {best_epoch}, val = {df.set_index('epoch').val.get(best_epoch, float('nan')):.6f}")
if df[["elbo", "reconst", "kl", "mse", "val"]].isna().any().any():
    print("ATTENTION : valeurs NaN dans le log.")
df.tail()

## 2. MSE : entraînement, validation, test

Échelle logarithmique ; l'axe de droite donne la RMSE en magnitudes. Le trait vertical marque le meilleur checkpoint (celui sauvé dans le `.h5`).

In [ ]:
sm = lambda s: s.rolling(SMOOTH, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.plot(df.epoch, df.mse, color="C0", alpha=0.25, lw=0.8)
ax.plot(df.epoch, sm(df.mse), color="C0", lw=2, label=f"train (lissé {SMOOTH} ep.)")
ax.plot(df.epoch, df.val, color="C1", alpha=0.25, lw=0.8)
ax.plot(df.epoch, sm(df.val), color="C1", lw=2, label=f"validation (lissé {SMOOTH} ep.)")
if len(test):
    ax.scatter(test.epoch, test.test, marker="s", s=14, color="C2", zorder=3, label="test (tous les 5 ep.)")
ax.axvline(best_epoch, color="k", ls="--", lw=1, label=f"meilleur checkpoint (ep. {best_epoch})")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE (unités normalisées)")
sec = ax.secondary_yaxis("right", functions=(lambda m: 2.5 * np.sqrt(np.maximum(m, 1e-12)), lambda r: (r / 2.5) ** 2))
sec.set_ylabel("RMSE équivalente (mag)")
ax.set_title("MSE de reconstruction"); ax.legend()
save(fig, "01_mse.png"); plt.show()

## 3. Perte, reconstruction et KL

- **ELBO (perte IWAE)** : la quantité minimisée.
- **Reconstruction** : `-log p(x|z)`, moyenne sur le batch.
- **KL** : divergence du posterior au prior ; en pointillés, le coefficient `kl_coef` d'annealing (0 pendant 10 epochs, puis rampe vers 1).

Un KL qui tombe vers 0 alors que le MSE stagne serait le signe d'un effondrement du posterior.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, col, title in [(axs[0], "elbo", "Perte (ELBO IWAE)"), (axs[1], "reconst", "Reconstruction  -log p(x|z)")]:
    ax.plot(df.epoch, df[col], alpha=0.3, color="C0", lw=0.8)
    ax.plot(df.epoch, sm(df[col]), color="C0", lw=2)
    ax.set_yscale("log"); ax.set_title(title); ax.set_xlabel("epoch")
    ax.axvline(best_epoch, color="k", ls="--", lw=1)

ax = axs[2]
ax.plot(df.epoch, df.kl, alpha=0.3, color="C3", lw=0.8)
ax.plot(df.epoch, sm(df.kl), color="C3", lw=2, label="KL moyen")
ax.set_title("KL et annealing"); ax.set_xlabel("epoch"); ax.set_ylabel("KL")
ax.axvline(best_epoch, color="k", ls="--", lw=1)
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(df.epoch, df.kl_coef, color="gray", ls=":", lw=1.5, label="kl_coef")
ax2.set_ylim(0, 1.05); ax2.set_ylabel("kl_coef")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper right")
save(fig, "02_perte_reconst_kl.png"); plt.show()

kl_peak = df.loc[df.kl.idxmax()]
print(f"KL max : {kl_peak.kl:.1f} (epoch {int(kl_peak.epoch)}) ; KL final (moy. 20 derniers ep.) : {df.kl.tail(20).mean():.1f}")

## 4. Diagnostic de surapprentissage

Si le MSE d'entraînement continue de baisser pendant que celui de validation remonte, c'est un surapprentissage.
On regarde les courbes lissées, le rapport val/train (seule sa **tendance** a un sens, voir l'avertissement en tête), le tableau par fenêtres et les pentes sur la seconde moitié de l'entraînement.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
ax = axs[0]
zoom = df[df.epoch >= max(30, int(0.12 * df.epoch.max()))]       # on enlève la phase de descente initiale
ib = int(np.argmin(np.abs(df.epoch.values - best_epoch)))
rel_tr = sm(df.mse) / sm(df.mse).iloc[ib]; rel_va = sm(df.val) / sm(df.val).iloc[ib]
ax.plot(zoom.epoch, rel_tr[zoom.index], color="C0", lw=2, label="train")
ax.plot(zoom.epoch, rel_va[zoom.index], color="C1", lw=2, label="validation")
ax.axhline(1, color="gray", lw=0.8); ax.axvline(best_epoch, color="k", ls="--", lw=1, label="meilleur checkpoint")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE lissé / MSE lissé au checkpoint")
ax.set_title("Plateau : MSE relatif au checkpoint (=1)"); ax.legend()

ax = axs[1]
ratio = sm(df.val) / sm(df.mse)
ax.plot(df.epoch, ratio, color="C4", lw=2)
ax.axvline(best_epoch, color="k", ls="--", lw=1)
ax.set_xlabel("epoch"); ax.set_ylabel("val / train (lissés)"); ax.set_title("Rapport val / train : la tendance compte, pas le niveau")
save(fig, "03_surapprentissage.png"); plt.show()

# tableau par fenêtres
g = df.assign(fenetre=(df.epoch - 1) // WINDOW).groupby("fenetre")
tab = g.agg(epochs_de=("epoch", "min"), epochs_a=("epoch", "max"), mse_train=("mse", "mean"),
            mse_val=("val", "mean"), val_std=("val", "std"), kl_moyen=("kl", "mean"), kl_coef=("kl_coef", "mean"))
tab["rmse_val_mag"] = 2.5 * np.sqrt(tab.mse_val)
display(tab.round(6))

# pentes sur la seconde moitié de l'entraînement
half = df[df.epoch >= df.epoch.max() // 2]
if len(half) > 20:
    sl = {c: np.polyfit(half.epoch, half[c], 1)[0] * 100 for c in ["mse", "val", "kl"]}
    print(f"Pentes sur les epochs >= {df.epoch.max() // 2} (par 100 epochs) :")
    print(f"  MSE train : {sl['mse']:+.2e}   MSE val : {sl['val']:+.2e}   KL : {sl['kl']:+.2f}")
    if sl["mse"] < 0 < sl["val"]:
        print("  -> train baisse et val monte : signature de surapprentissage.")
    elif sl["mse"] < 0 and sl["val"] <= 0:
        print("  -> les deux baissent encore : le modèle apprend toujours, un entraînement plus long peut aider.")
    else:
        print("  -> pas de tendance claire de surapprentissage.")

## 5. Poids relatif du KL

Rapport `kl_coef × KL / reconstruction` (approximatif : la perte réelle est une borne IWAE, pas une simple somme).
Il indique si le terme de régularisation pèse réellement dans l'optimisation.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
rel = df.kl_coef * df.kl / df.reconst
ax.plot(df.epoch, rel, alpha=0.3, color="C5", lw=0.8)
ax.plot(df.epoch, sm(rel), color="C5", lw=2)
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("kl_coef · KL / reconstruction")
ax.set_title("Poids relatif du terme KL (approx.)")
save(fig, "04_poids_kl.png"); plt.show()
print(f"Poids relatif du KL : {rel.iloc[-20:].mean():.2%} en fin d'entraînement (moyenne des 20 derniers epochs).")

## 6. Choix du meilleur checkpoint : bruit de la validation

Le checkpoint est le **minimum** de `val_metric` sur tous les epochs. Comme la validation est bruitée (un seul tirage latent par courbe),
ce minimum est un peu optimiste : en ré-évaluant le même modèle, on retombe plutôt sur la moyenne locale (« malédiction du gagnant »).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(df.epoch, df.val, ".", ms=3, color="C1", alpha=0.5, label="val_metric (chaque epoch)")
ax.plot(df.epoch, df.val.cummin(), color="k", lw=1.5, label="minimum courant")
ax.scatter([best_epoch], [df.set_index("epoch").val[best_epoch]], color="r", zorder=4, label=f"checkpoint (ep. {best_epoch})")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE validation"); ax.legend()
ax.set_title("Meilleur checkpoint et bruit de la validation")
save(fig, "05_checkpoint.png"); plt.show()

v = df.set_index("epoch").val
win = v[(v.index >= best_epoch - 30) & (v.index <= best_epoch + 30)]
z = (win.mean() - v[best_epoch]) / win.std()
print(f"val au checkpoint : {v[best_epoch]:.6f}")
print(f"moyenne ± écart-type sur ±30 epochs : {win.mean():.6f} ± {win.std():.6f}")
print(f"Le checkpoint est {z:.1f} σ sous la moyenne locale -> attendre en ré-évaluation une valeur plus proche de {win.mean():.6f}.")

## 7. *(Optionnel)* Analyse par courbe, à partir de `evaluate_unsupervised_lsst.py`

Nécessite, dans le dossier courant, pour chaque split (`val`, `test`) :
`evaluate_<split>_decoded_lcs_<TAG>.npz` et `evaluate_<split>_objIds_dataloader_<TAG>.npy`
(produits par `python evaluate_unsupervised_lsst.py --tag <TAG> --setting val|test`). La section est ignorée si les fichiers sont absents.

In [ ]:
BANDS = ["g", "r", "i", "z"]
rng = np.random.default_rng(0)

def load_eval(split):
    fz = f"evaluate_{split}_decoded_lcs_{TAG}.npz"; fi = f"evaluate_{split}_objIds_dataloader_{TAG}.npy"
    if not (os.path.exists(fz) and os.path.exists(fi)):
        return None
    z = np.load(fz); ids = np.load(fi)
    keys = sorted(z.files, key=lambda k: int(k.split("_")[1]))
    mse_c, npts, sq_b, n_b = [], [], None, None
    for k in keys:
        pred, obs, mask = z[k][0], z[k][1], z[k][2]        # (T, dim)
        err = (pred - obs) ** 2 * mask
        if sq_b is None:
            sq_b, n_b = np.zeros(pred.shape[-1]), np.zeros(pred.shape[-1])
        sq_b += err.sum(0); n_b += mask.sum(0)
        mse_c.append(err.sum() / mask.sum()); npts.append(mask.sum())
    return dict(ids=ids, mse=np.array(mse_c), n=np.array(npts), sq_b=sq_b, n_b=n_b)

ev = {s: load_eval(s) for s in ["val", "test"]}
ev = {s: d for s, d in ev.items() if d is not None}
print("Splits disponibles :", list(ev) or "aucun (section ignorée)")

In [ ]:
if ev:
    def boot(x, n=5000):
        return np.array([rng.choice(x, len(x)).mean() for _ in range(n)])
    colors = {"val": "C1", "test": "C2"}
    fig, axs = plt.subplots(2, 2, figsize=(13, 9))

    # (a) distribution de la RMSE par courbe, en mag
    ax = axs[0, 0]
    allr = np.concatenate([2.5 * np.sqrt(d["mse"]) for d in ev.values()])
    bins = np.logspace(np.log10(max(allr.min(), 1e-3)), np.log10(allr.max()), 40)
    for s, d in ev.items():
        r = 2.5 * np.sqrt(d["mse"])
        ax.hist(r, bins=bins, alpha=0.55, color=colors[s], label=f"{s} (médiane {np.median(r):.3f} mag)")
        ax.axvline(np.median(r), color=colors[s], ls="--")
    ax.set_xscale("log"); ax.set_xlabel("RMSE par courbe (mag)"); ax.set_ylabel("nb de courbes"); ax.legend()
    ax.set_title("Distribution de l'erreur par courbe")

    # (b) RMSE par bande
    ax = axs[0, 1]
    w = 0.38
    for j, (s, d) in enumerate(ev.items()):
        rb = 2.5 * np.sqrt(d["sq_b"] / np.maximum(d["n_b"], 1))
        x = np.arange(len(rb)) + (j - (len(ev) - 1) / 2) * w
        ax.bar(x, rb, width=w, color=colors[s], label=s)
        for xi, v, nb in zip(x, rb, d["n_b"]):
            ax.text(xi, v, f"{int(nb)}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(np.arange(len(rb))); ax.set_xticklabels(BANDS[:len(rb)])
    ax.set_ylabel("RMSE (mag)"); ax.set_title("RMSE par bande (au-dessus : nb de points)"); ax.legend()

    # (c) erreur vs nombre de points
    ax = axs[1, 0]
    for s, d in ev.items():
        ax.scatter(d["n"], d["mse"], s=10, alpha=0.5, color=colors[s], label=s)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("nb de points observés"); ax.set_ylabel("MSE par courbe")
    ax.set_title("Erreur vs nombre de points"); ax.legend()

    # (d) CDF
    ax = axs[1, 1]
    for s, d in ev.items():
        r = np.sort(2.5 * np.sqrt(d["mse"]))
        ax.plot(r, np.arange(1, len(r) + 1) / len(r), color=colors[s], label=s)
    ax.set_xscale("log"); ax.set_xlabel("RMSE par courbe (mag)"); ax.set_ylabel("fraction cumulée")
    ax.set_title("Fonction de répartition"); ax.legend()
    save(fig, "06_par_courbe.png"); plt.show()

    for s, d in ev.items():
        b = boot(d["mse"])
        print(f"{s:5s}: MSE moyen par courbe = {d['mse'].mean():.5f}  IC95% [{np.percentile(b, 2.5):.5f}, {np.percentile(b, 97.5):.5f}]"
              f" ; RMSE médiane par courbe = {np.median(2.5 * np.sqrt(d['mse'])):.3f} mag")
    if len(ev) == 2:
        a, c = ev["val"]["mse"], ev["test"]["mse"]
        dd = np.array([rng.choice(c, len(c)).mean() - rng.choice(a, len(a)).mean() for _ in range(5000)])
        print(f"Différence test - val : {c.mean() - a.mean():+.5f}  IC95% [{np.percentile(dd, 2.5):+.5f}, {np.percentile(dd, 97.5):+.5f}]"
              "  (si l'IC contient 0, l'écart n'est pas significatif)")
    worst = {s: np.argsort(d["mse"])[::-1][:5] for s, d in ev.items()}
    for s, d in ev.items():
        print(f"\n5 pires courbes ({s}) :")
        for i in worst[s]:
            print(f"  {d['ids'][i]}   MSE = {d['mse'][i]:.4f}   points = {int(d['n'][i])}")

## 8. Récapitulatif

In [ ]:
v = df.set_index("epoch").val
print(f"Log : {LOG_FILE}")
print(f"Epochs : {int(df.epoch.max())} | meilleur checkpoint : epoch {best_epoch} (val = {v[best_epoch]:.6f}, RMSE ≈ {2.5*np.sqrt(v[best_epoch]):.3f} mag)")
print(f"MSE train final (moy. 20 derniers ep.) : {df.mse.tail(20).mean():.6f} | val : {df.val.tail(20).mean():.6f}")
print(f"KL final (moy. 20 derniers ep.) : {df.kl.tail(20).mean():.1f}")
print("Figures enregistrées dans :", os.path.abspath(FIG_DIR))